# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Kilifi County Mental Health survey dataset using the `mlcroissant` library, following the Croissant standard. All dataset entities are referenced explicitly by their `@id`.

### Dataset Source
The dataset metadata is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as a json dict
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets and their fields by iterating through record sets, and displaying their field and column `@id`s.

**Note:** All references are by `@id` as per Croissant best practices.

In [ ]:
# List all record sets in the dataset
print('Available record sets:')
record_sets = []
for record_set in dataset.metadata.record_sets:
    print(f"  @id: {record_set.id} | Name: {record_set.name}")
    record_sets.append(record_set.id)

# Display fields and columns for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nRecord set `@id`: {rs.id}")
    print(f"Fields: [@id]")
    for field in rs.fields:
        print(f"    {field.id}")
    print(f"Columns: [@id]")
    if rs.columns:
        for col in rs.columns:
            print(f"    {col.id}")

## 3. Data Extraction
Load the data from each record set into a pandas DataFrame. Record set and field `@id`s should be used for referencing.


In [ ]:
dfs = {}
# Iterate and extract each record set
for record_set_id in record_sets:
    print(f"Extracting data for record set: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    if len(recs) > 0:
        df = pd.DataFrame(recs)
        dfs[record_set_id] = df
        print(f"  DataFrame shape: {df.shape}\n  Columns (@id): {list(df.columns)}")
    else:
        print("  No records found.")

# Choose the main survey record set. The main CSV is usually at .../records, and its @id is likely the only non-empty record set.
main_record_set_id = None
for k, v in dfs.items():
    if v.shape[0] > 0:
        main_record_set_id = k
        break

print(f"\nMain analysis record set will be: {main_record_set_id}")
# Show the first few records
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping.

Let's choose a numeric mental health score from the available fields, e.g., PHQ-9 or GAD-7 scores, and group by a demographic attribute, e.g., `level_of_education`.

> **Tip:** The actual field `@id` for these variables can be found in the overview above. Substitute them accordingly.

In [ ]:
# You may need to adjust field @id names to match exact output from overview above.
# For example, phq_9_total_score or gad_7_total_score for numeric; level_of_education for group.

# Assign field IDs (adjust as needed based on printed columns from above)
numeric_field_id = 'phq_9_total_score'  # example, replace with exact @id from your schema
group_field_id = 'level_of_education'  # example, replace with exact @id from your schema

# Reference DataFrame
df = dfs[main_record_set_id]

# Filter for score > 10
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} of {len(df)}")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if present
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} by {group_field_id}:\n", grouped)
else:
    print(f"Field '{numeric_field_id}' not in the DataFrame columns. Adjust field @id as per the schema.")

## 5. Visualization
Visualize the distribution of a numeric score and its relationship with a demographic variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
# Boxplot by educational level
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² Kilifi County Mental Health Survey dataset using the `mlcroissant` library. We've demonstrated how to reference data entities by their `@id`, inspect schema organization, extract records, and perform basic exploratory and visual analysis on survey scores by demographic groups. For further work, you might apply advanced statistics or model-building, leveraging the AI-ready structure provided by the Croissant schema.